<a href="https://colab.research.google.com/github/OshankKarn/DNNAssignment2/blob/main/Problem2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNN Assignment 2 - Problem 2: Transformer from scratch
**Tasks 2.1-2.2 (this notebook part):** attention, multi-head attention, sinusoidal positions and the pre-LN encoder block.
Allowed: `nn.Linear`, `nn.LayerNorm`, `nn.Embedding`, `nn.Dropout`. Not allowed: `nn.MultiheadAttention`, `nn.Transformer*`.
`F.scaled_dot_product_attention` is used **only** inside tests.

Student ID: `<fill in>`

In [1]:
import os, math, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

os.makedirs('figures', exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(0)
print('device:', DEVICE, '| torch', torch.__version__)

device: cuda | torch 2.14.1+cu130


## Task 1 - Scaled dot-product attention

In [2]:
def scaled_dot_product_attention(q, k, v, mask=None):
    # q: (..., Tq, dk)  k: (..., Tk, dk)  v: (..., Tk, dv)
    # mask: bool, broadcastable to (..., Tq, Tk); True = may attend, False = blocked
    dk = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(dk)      # (..., Tq, Tk)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)                # rows sum to 1
    return weights @ v, weights                            # out: (..., Tq, dv)

def causal_mask(T, device=None):
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))   # (T, T), lower triangle True

In [3]:
# Unit tests (direct asserts)
set_seed(0)
q, k, v = torch.randn(2, 4, 6, 8), torch.randn(2, 4, 6, 8), torch.randn(2, 4, 6, 8)   # (B, H, T, dk)

out, w = scaled_dot_product_attention(q, k, v)
assert torch.allclose(w.sum(-1), torch.ones(2, 4, 6), atol=1e-6), 'rows must sum to 1'
assert (out - F.scaled_dot_product_attention(q, k, v)).abs().max() < 1e-5, 'no-mask mismatch'

cm = causal_mask(6)
out_c, w_c = scaled_dot_product_attention(q, k, v, cm)
assert (out_c - F.scaled_dot_product_attention(q, k, v, is_causal=True)).abs().max() < 1e-5, 'causal mismatch'
assert (w_c.masked_fill(cm, 0.0) == 0).all(), 'future positions must get exactly zero weight'
assert torch.allclose(w_c.sum(-1), torch.ones(2, 4, 6), atol=1e-6)
print('attention tests passed')

attention tests passed


### Hand-checkable 3-token example
**Replace with the instructor's example if one is supplied.** Mine: $d_k=4$ so $\sqrt{d_k}=2$.
$Q=K=\begin{pmatrix}2&0&0&0\\0&2&0&0\\0&0&2&0\end{pmatrix}$, $V=\begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$.

Scores $=QK^\top/2$ have 2 on the diagonal and 0 elsewhere, so row 1 of the weights is $\left(\tfrac{e^2}{e^2+2},\tfrac{1}{e^2+2},\tfrac{1}{e^2+2}\right)$ (rows 2 and 3 are the same with the large entry moved), and output row 1 is $\tfrac{1}{e^2+2}\,(e^2+1,\;2)$ ... work this out yourself and show the steps in the report.

In [4]:
Q = K = torch.tensor([[2., 0, 0, 0], [0, 2., 0, 0], [0, 0, 2., 0]])
V = torch.tensor([[1., 0], [0, 1.], [1., 1.]])
out, w = scaled_dot_product_attention(Q, K, V)

# By hand (pure Python, no torch): row i of weights = softmax with e^2 on the diagonal
e2 = math.exp(2); S = e2 + 2
W_hand = torch.tensor([[e2 / S if j == i else 1 / S for j in range(3)] for i in range(3)])
V_l = V.tolist()
O_hand = torch.tensor([[sum(W_hand[i, j].item() * V_l[j][c] for j in range(3)) for c in range(2)] for i in range(3)])
assert torch.allclose(w, W_hand, atol=1e-6) and torch.allclose(out, O_hand, atol=1e-6)
print(w); print(out)

tensor([[0.7870, 0.1065, 0.1065],
        [0.1065, 0.7870, 0.1065],
        [0.1065, 0.1065, 0.7870]])
tensor([[0.8935, 0.2130],
        [0.2130, 0.8935],
        [0.8935, 0.8935]])


## Task 2 - Multi-head attention, positional encoding, encoder block

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, n_heads, dropout=0.0):
        super().__init__()
        assert d % n_heads == 0
        self.d, self.H, self.dh = d, n_heads, d // n_heads
        self.W_q, self.W_k, self.W_v, self.W_o = (nn.Linear(d, d) for _ in range(4))
        self.drop = nn.Dropout(dropout)
    def split(self, x):                       # (B, T, d) -> (B, H, T, dh)
        B, T, _ = x.shape
        return x.view(B, T, self.H, self.dh).transpose(1, 2)
    def forward(self, x, mask=None, return_attn=False):
        q, k, v = self.split(self.W_q(x)), self.split(self.W_k(x)), self.split(self.W_v(x))   # (B, H, T, dh)
        out, w = scaled_dot_product_attention(q, k, v, mask)       # out (B, H, T, dh), w (B, H, T, T)
        out = out.transpose(1, 2).reshape(x.size(0), x.size(1), self.d)   # concat heads: (B, T, d)
        out = self.W_o(self.drop(out))
        return (out, w) if return_attn else out

class SinusoidalPE(nn.Module):
    def __init__(self, d, max_len=4096):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)                                  # (L, 1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))        # (d/2,)
        pe = torch.zeros(max_len, d)
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, x):                      # x: (B, T, d)
        return x + self.pe[:x.size(1)]

class EncoderBlock(nn.Module):
    # pre-LayerNorm: x = x + Drop(MHA(LN(x))); x = x + Drop(FFN(LN(x)))
    def __init__(self, d, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, n_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mask=None, return_attn=False):
        a = self.attn(self.ln1(x), mask, return_attn)
        if return_attn: a, w = a
        x = x + self.drop(a)
        x = x + self.drop(self.ff(self.ln2(x)))
        return (x, w) if return_attn else x

In [6]:
# Tests: shapes, positional encoding values, causality, mask-free permutation behaviour
set_seed(0)
d, H, B, T = 128, 4, 3, 10
x = torch.randn(B, T, d)

mha = MultiHeadAttention(d, H).eval()
y, w = mha(x, causal_mask(T), return_attn=True)
assert y.shape == (B, T, d) and w.shape == (B, H, T, T)

pe = SinusoidalPE(d)
assert torch.allclose(pe.pe[0, 0::2], torch.zeros(d // 2)) and torch.allclose(pe.pe[0, 1::2], torch.ones(d // 2))
assert abs(pe.pe[1, 0].item() - math.sin(1.0)) < 1e-6          # PE(1, 0) = sin(1)

blk = EncoderBlock(d, H, 4 * d, dropout=0.1).eval()
assert blk(x).shape == x.shape

# causal block: changing future tokens must not change earlier outputs
x2 = x.clone(); x2[:, 6:] = torch.randn(B, T - 6, d)
m = causal_mask(T)
assert torch.allclose(blk(x, m)[:, :6], blk(x2, m)[:, :6], atol=1e-5), 'causal leak'
assert not torch.allclose(blk(x)[:, :6], blk(x2)[:, :6], atol=1e-5)   # without a mask it does leak
print('multi-head / PE / block tests passed')

multi-head / PE / block tests passed


In [7]:
# Parameter count: derivation vs sum(p.numel())
d, d_ff = 128, 512
blk = EncoderBlock(d, 4, d_ff)
total = sum(p.numel() for p in blk.parameters())

w_attn = 4 * d * d                  # W_Q, W_K, W_V, W_O weights
w_ff   = 2 * d * d_ff               # two feed-forward matrices
w_only = w_attn + w_ff              # 4d^2 + 2*d*d_ff
b_attn = 4 * d                      # one bias per attention projection
b_ff   = d_ff + d                   # biases of the two FFN linears
ln     = 2 * (2 * d)                # two LayerNorms x (gamma + beta)
print(pd.DataFrame({'part': ['attention weights', 'FFN weights', 'attention biases', 'FFN biases', 'LayerNorm (2x)', 'TOTAL'],
                    'count': [w_attn, w_ff, b_attn, b_ff, ln, w_only + b_attn + b_ff + ln]}))
assert w_only == 196_608
assert total == w_only + b_attn + b_ff + ln == 198_272
print('block parameters:', total)

                part   count
0  attention weights   65536
1        FFN weights  131072
2   attention biases     512
3         FFN biases     640
4     LayerNorm (2x)     512
5              TOTAL  198272
block parameters: 198272


**Write-up for the report (your own words):** attention contributes $4d^2$ weights (four $d\times d$ projections) and the feed-forward layer $2\,d\,d_{ff}$, so $4d^2+2d\,d_{ff}=65{,}536+131{,}072=196{,}608$ for $d=128$, $d_{ff}=512$. Biases add $4d=512$ (attention) $+\,d_{ff}+d=640$ (FFN), and two LayerNorms add $2\cdot2d=512$, giving $198{,}272$ in total. Note that positional encoding has no parameters (a fixed buffer) and the number of heads does not change the count.